##### date_format()
- is used to **convert** a **date, timestamp or string** column into a **string** value in a **specified format**.

- works on **input columns** that are already a **DateType, TimestampType or a valid string** that `Spark` can implicitly **cast to a date/timestamp**.

- If your **input string** is in a **non-standard format (e.g., MM/dd/yyyy instead of the default yyyy-MM-dd)**, you should first use the **to_date() or to_timestamp()** functions to parse it correctly into a **Date or Timestamp type**, and then apply **date_format()** to convert it to the desired output **string format**.

##### Syntax

     date_format(column, format)

| Argument |   Description                                                                         |
|----------|---------------------------------------------------------------------------------------|
|  column  | The input column **(DateType, TimestampType or StringType)** that you want to format. |
|  format  | A **string** literal representing the desired output format using **Datetime**.       |
|  Output  | is **StringType**, **not DateType**.                                                  |

**Common formats:**

|         Year-Month             |        ISO-like format          |          Time only            |       custom format                 |
|--------------------------------|---------------------------------|-------------------------------|-------------------------------------|
| date_format("date", "yyyy-MM") | date_format("date", "yyyyMMdd") | date_format("ts", "HH:mm:ss") |  date_format("date", "dd/MM/yyyy")  |

**date_format() vs to_date()**

| Function        |   Output   |     Purpose             |
| --------------- | ---------- | ----------------------- |
| `date_format()` | **String** | **Formatting**          |
| `to_date()`     | **Date**   | **DataType conversion** |

❌ This is wrong if you need a date:

     date_format("date", "yyyy-MM-dd")

✅ Use this instead:

     to_date("date")

**Content:**
- `Convert date to string`
- `Convert timestamp to string`
- `Convert string date to string`
- `Convert non-standard string date format`
- `Extract parts of a date`
- `Month name and day name`
- `Format current_date columns`

In [0]:
from pyspark.sql.functions import col, date_format, to_date

##### 1) Convert `date to string`

In [0]:
# Create a sample DataFrame with a date column in default 'yyyy-MM-dd' format
data = [("2026-01-28",),
        ("2025-12-05",),
        ("2023-12-21",),
        ("2022-01-01",),
        ("2021-01-01",),
        ("2020-01-01",),
        ("2019-01-01",)]

df = spark.createDataFrame(data, ["start_date"])
df_frmt = df.withColumn("frmt_date", col("start_date").cast("date"))
display(df_frmt)

start_date,frmt_date
2026-01-28,2026-01-28
2025-12-05,2025-12-05
2023-12-21,2023-12-21
2022-01-01,2022-01-01
2021-01-01,2021-01-01
2020-01-01,2020-01-01
2019-01-01,2019-01-01


In [0]:
df_frmt_01 = (df_frmt \
    .select("frmt_date", 
            date_format("frmt_date", "dd/MM/yyyy").alias("formatted_date"),
            date_format("frmt_date", "dd-MM-yyyy").alias("formatted_date"),
            date_format("frmt_date", "MMM-yyyy").alias("order_month"),              # Display frmt_date as Jan-2026
            date_format("frmt_date", "dd-MMM-yy").alias("order_date"),
            date_format("frmt_date", "yyyy MM dd").alias("format_01"),
            date_format("frmt_date", "yyyy MMM dd").alias("format_02"),
            date_format("frmt_date", "yyyy MMMM dd E").alias("format_03"),
            date_format("frmt_date", "MMMM dd, yyyy").alias("format_04"),           # Format to a more descriptive string
            date_format("frmt_date", "yyyy-MM-dd HH:mm:ss").alias("format_05")))        

display(df_frmt_01)

frmt_date,formatted_date,formatted_date,order_month,order_date,format_01,format_02,format_03,format_04,format_05
2026-01-28,28/01/2026,28-01-2026,Jan-2026,28-Jan-26,2026 01 28,2026 Jan 28,2026 January 28 Wed,"January 28, 2026",2026-01-28 00:00:00
2025-12-05,05/12/2025,05-12-2025,Dec-2025,05-Dec-25,2025 12 05,2025 Dec 05,2025 December 05 Fri,"December 05, 2025",2025-12-05 00:00:00
2023-12-21,21/12/2023,21-12-2023,Dec-2023,21-Dec-23,2023 12 21,2023 Dec 21,2023 December 21 Thu,"December 21, 2023",2023-12-21 00:00:00
2022-01-01,01/01/2022,01-01-2022,Jan-2022,01-Jan-22,2022 01 01,2022 Jan 01,2022 January 01 Sat,"January 01, 2022",2022-01-01 00:00:00
2021-01-01,01/01/2021,01-01-2021,Jan-2021,01-Jan-21,2021 01 01,2021 Jan 01,2021 January 01 Fri,"January 01, 2021",2021-01-01 00:00:00
2020-01-01,01/01/2020,01-01-2020,Jan-2020,01-Jan-20,2020 01 01,2020 Jan 01,2020 January 01 Wed,"January 01, 2020",2020-01-01 00:00:00
2019-01-01,01/01/2019,01-01-2019,Jan-2019,01-Jan-19,2019 01 01,2019 Jan 01,2019 January 01 Tue,"January 01, 2019",2019-01-01 00:00:00


**Pattern cheat-sheet**

| Pattern |         Meaning                |   Example   |
| ------- | -------------------------------|-------------|
| yyyy    | Year                           |    2024     |
| MM      | Month number (zero-padded)     |    02       |
| MMM     | Month short name               |    Jan      |
| MMMM    | Month full name                |    January  |
| dd      | Day of the month (zero-padded) |    05       |
| E / EEE | Day short name                 |    Mon      | 
| EEEE    | Full day of the week           |    Monday   |
| HH      | Hour in 24-hour format         |             |
| hh      | Hour in 12-hour format         |             |
| mm      | Minutes                        |             |
| ss      | Seconds                        |             |

##### 2) Convert `timestamp to string`

In [0]:
data = [("2026-01-28 14:35:40",),
        ("2026-01-28 14:35:40",),
        ("2026-01-28 14:35:40",),
        ("2026-01-28 14:35:40",),
        ("2026-01-28 14:35:40",),
        ("2026-01-28 14:35:40",)]

df_ts = spark.createDataFrame(data, ["ts"]).withColumn("ts_ts", col("ts").cast("timestamp"))
display(df_ts)

ts,ts_ts
2026-01-28 14:35:40,2026-01-28T14:35:40.000Z
2026-01-28 14:35:40,2026-01-28T14:35:40.000Z
2026-01-28 14:35:40,2026-01-28T14:35:40.000Z
2026-01-28 14:35:40,2026-01-28T14:35:40.000Z
2026-01-28 14:35:40,2026-01-28T14:35:40.000Z
2026-01-28 14:35:40,2026-01-28T14:35:40.000Z


In [0]:
df_frmt_03 = df_ts \
    .select("ts_ts", date_format("ts_ts", "dd-MM-yyyy HH:mm:ss").alias("formatted_ts_01"),
                     date_format("ts_ts", "MM/dd/yyyy hh:mm").alias("formatted_ts_02"),
                     date_format("ts_ts", "yyyy/MM/dd HH:mm:ss").alias("formatted_ts_03"),
                     date_format("ts_ts", "dd/MM/yyyy H").alias("formatted_ts_04"),
                     date_format("ts_ts", "dd/MM/yyyy H:mm").alias("formatted_ts_05"),
                     date_format("ts_ts", "yyyy/MM/dd HH:mm:ss").alias("formatted_ts_06"))
    
display(df_frmt_03)

ts_ts,formatted_ts_01,formatted_ts_02,formatted_ts_03,formatted_ts_04,formatted_ts_05,formatted_ts_06
2026-01-28T14:35:40.000Z,28-01-2026 14:35:40,01/28/2026 02:35,2026/01/28 14:35:40,28/01/2026 14,28/01/2026 14:35,2026/01/28 14:35:40
2026-01-28T14:35:40.000Z,28-01-2026 14:35:40,01/28/2026 02:35,2026/01/28 14:35:40,28/01/2026 14,28/01/2026 14:35,2026/01/28 14:35:40
2026-01-28T14:35:40.000Z,28-01-2026 14:35:40,01/28/2026 02:35,2026/01/28 14:35:40,28/01/2026 14,28/01/2026 14:35,2026/01/28 14:35:40
2026-01-28T14:35:40.000Z,28-01-2026 14:35:40,01/28/2026 02:35,2026/01/28 14:35:40,28/01/2026 14,28/01/2026 14:35,2026/01/28 14:35:40
2026-01-28T14:35:40.000Z,28-01-2026 14:35:40,01/28/2026 02:35,2026/01/28 14:35:40,28/01/2026 14,28/01/2026 14:35,2026/01/28 14:35:40
2026-01-28T14:35:40.000Z,28-01-2026 14:35:40,01/28/2026 02:35,2026/01/28 14:35:40,28/01/2026 14,28/01/2026 14:35,2026/01/28 14:35:40


##### 3) Convert `default string date to string`

In [0]:
df_strdate = spark.createDataFrame([('2025-05-25','2026-02-26')], schema=['Str_dt1','Str_dt2'])
display(df_strdate)

Str_dt1,Str_dt2
2025-05-25,2026-02-26


In [0]:
df_strdate_frmt = df_strdate.withColumn('frmt_Str_dt1', date_format(col("Str_dt1"), 'dd-MM-yyyy'))\
                            .withColumn('frmt_Str_dt2', date_format(col("Str_dt2"), 'dd-MM-yyyy'))\

display(df_strdate_frmt)

Str_dt1,Str_dt2,frmt_Str_dt1,frmt_Str_dt2
2025-05-25,2026-02-26,25-05-2025,26-02-2026


##### 4) Convert `custom string date to string`

In [0]:
df_todate = spark.createDataFrame([('05/22/2022','10/21/2022')], schema=['Date_col1','Date_col2'])
display(df_todate)

Date_col1,Date_col2
05/22/2022,10/21/2022


In [0]:
df_todate_frmt = df_todate.withColumn('date_col1', date_format(to_date(col("Date_col1"), "MM/dd/yyyy"), 'dd-MM-yyyy'))\
                          .withColumn('date_col2', date_format(to_date(col("Date_col2"), "MM/dd/yyyy"), 'dd-MM-yyyy'))\

display(df_todate_frmt)

date_col1,date_col2
22-05-2022,21-10-2022


##### 5) Extract parts of a date

In [0]:
df_frmt_03 = df_frmt.select("frmt_date",
    date_format("frmt_date", "yyyy").alias("year"),
    date_format("frmt_date", "MM").alias("month"),
    date_format("frmt_date", "dd").alias("day")
)

display(df_frmt_03)

frmt_date,year,month,day
2026-01-28,2026,01,28
2025-12-05,2025,12,05
2023-12-21,2023,12,21
2022-01-01,2022,01,01
2021-01-01,2021,01,01
2020-01-01,2020,01,01
2019-01-01,2019,01,01


##### 6) Month name and day name

In [0]:
df_frmt_04 = df_frmt.select("frmt_date",
    date_format("frmt_date", "MM").alias("month_number"),
    date_format("frmt_date", "MMM").alias("month_short_name"),    
    date_format("frmt_date", "MMMM").alias("month_full_name"),
    date_format("frmt_date", "dd").alias("day"),
    date_format("frmt_date", "EEE").alias("day_short_name"),
    date_format("frmt_date", "EEEE").alias("day_full_name")
)

display(df_frmt_04)

frmt_date,month_number,month_short_name,month_full_name,day,day_short_name,day_full_name
2026-01-28,01,Jan,January,28,Wed,Wednesday
2025-12-05,12,Dec,December,05,Fri,Friday
2023-12-21,12,Dec,December,21,Thu,Thursday
2022-01-01,01,Jan,January,01,Sat,Saturday
2021-01-01,01,Jan,January,01,Fri,Friday
2020-01-01,01,Jan,January,01,Wed,Wednesday
2019-01-01,01,Jan,January,01,Tue,Tuesday


##### 7) Format `current_date` columns

In [0]:
# Create DataFrame
df = spark.createDataFrame([["1"]], ["id"])
display(df)

id
1


In [0]:
from pyspark.sql.functions import to_date, current_date, current_timestamp, date_format

# Using date_format()
df_ctst = (df.select(current_date().alias("current_date"),
      date_format(current_timestamp(),"yyyy MM dd").alias("yyyy MM dd"),
      date_format(current_timestamp(),"MM/dd/yyyy hh:mm").alias("MM/dd/yyyy"),
      date_format(current_timestamp(),"yyyy MMM dd").alias("yyyy MMMM dd"),
      date_format(current_timestamp(),"yyyy MMMM dd E").alias("yyyy MMMM dd E"))
)
   
display(df_ctst)

current_date,yyyy MM dd,MM/dd/yyyy,yyyy MMMM dd,yyyy MMMM dd E
2026-03-02,2026 03 02,03/02/2026 06:01,2026 Mar 02,2026 March 02 Mon
